# Notebook 14 — Robustez frente a Ataques Adaptativos

**¿Qué pasa si el atacante conoce el detector?** Toda la evaluación anterior (NB03–NB10) asume un **atacante ciego**: inyecta ataques con severidades aleatorias y la severidad no depende del detector. En la práctica, un atacante razonable adapta su intensidad para **maximizar el daño económico** mientras **minimiza la probabilidad de ser detectado**.

**Marco experimental:**
- Para cada detector base, buscamos el **umbral mínimo de severidad** (`epsilon*`) por encima del cual el detector marca el ataque, y por debajo del cual el ataque pasa desapercibido.
- Repetimos por cada tipo de ataque (`scaling`, `offset`, `noise`, `ramp`, `step`, `replay`).
- Construimos la curva **detection rate vs severidad** — fundamental para discutir resistencia.

**Búsqueda:**
- **Búsqueda binaria** sobre la magnitud del ataque (rápido, ~10 iteraciones).
- Repetida 30 veces por (modelo, tipo de ataque) para obtener media y desviación.

**Lectura del resultado:**
- Si la *minimum-evasion budget* es **muy pequeña** (~5-10% de fracción de robo), el detector es frágil.
- Si es **grande** (>30%), el detector es robusto: cualquier ataque suficientemente "valioso" para el atacante es detectable.

Este capítulo cierra el TFG con la pregunta más exigente del tribunal: *"¿y si el atacante es inteligente?"*

**Apple Silicon nota:** los detectores Keras se cargan en memoria; cada búsqueda binaria son ~10 forward passes sobre 100-180 timesteps → trivialmente rápido en Metal.


## 0. Setup

In [1]:
import os, json, time, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
print('OK')

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
tf.keras.mixed_precision.set_global_policy('float32')


OK


## 1. Carga de Datos + modelos

In [2]:
BASE_DIR = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/'
DATA_DIR = os.path.join(BASE_DIR, 'data')

with open(os.path.join(DATA_DIR, 'pipeline_config.json')) as f:
    config = json.load(f)

FEATURE_COLS = config['feature_cols']
ATTACK_TYPES = config['attack_types']

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_clean.csv'),
                       index_col='datetime', parse_dates=True)
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_with_attacks.csv'),
                       index_col='datetime', parse_dates=True)
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_with_attacks.csv'),
                       index_col='datetime', parse_dates=True)

print('DATOS CARGADOS (UCI real)')
print('=' * 65)
for nm, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos = (d['label']==1).sum() if 'label' in d.columns else 0
    print(f'  {nm:6s} {len(d):>9,d}   ataques: {pos:>7,d}')

def compute_features(df, mode='full'):
    f = df[FEATURE_COLS].copy()
    f['VI_residual'] = (df['Global_active_power']
                        - df['Voltage'] * df['Global_intensity'] / 1000.0)
    if mode in ('medium','full'):
        h = df.index.hour + df.index.minute / 60.0
        f['hour_sin'] = np.sin(2*np.pi*h/24); f['hour_cos'] = np.cos(2*np.pi*h/24)
        d = df.index.dayofweek
        f['dow_sin']  = np.sin(2*np.pi*d/7);  f['dow_cos']  = np.cos(2*np.pi*d/7)
        f['gap_diff1']= df['Global_active_power'].diff().fillna(0)
    if mode == 'full':
        f['vi_res_abs'] = f['VI_residual'].abs()
        f['vi_res_roll15_mean'] = f['vi_res_abs'].rolling(15, min_periods=1).mean()
        f['gap_intensity_ratio']= df['Global_active_power'] / (df['Global_intensity']+0.01)
        f['sm_gap_ratio']=((df['Sub_metering_1']+df['Sub_metering_2']+df['Sub_metering_3'])/1000.0
                            / (df['Global_active_power']+0.01))
    return f.fillna(0)

# Solo evaluamos sobre TEST (donde tenemos ataques inyectados)
y_test = df_test['label'].values

# Cargar Dense-AE (el modelo principal del TFG)
scaler_dae  = joblib.load(os.path.join(DATA_DIR, 'scaler_dense_autoencoder.pkl'))
weights_dae = np.load(os.path.join(DATA_DIR, 'feature_weights_dense_ae.npy'))
threshold_dae = float(np.load(os.path.join(DATA_DIR, 'threshold_dense_ae.npy'))[0])
dae = load_model(os.path.join(DATA_DIR, 'model_dense_autoencoder.keras'))
print(f'Dense-AE cargado. threshold = {threshold_dae:.4f}')


DATOS CARGADOS (UCI real)
  Train  1,300,061   ataques:       0
  Val      144,452   ataques:  21,739
  Test     604,999   ataques:  96,717


2026-05-20 16:53:46.538473: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2026-05-20 16:53:46.538502: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-20 16:53:46.538507: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1779288826.538520  171518 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1779288826.538557  171518 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Dense-AE cargado. threshold = 0.2891


## 2. Función de scoring para Dense-AE

In [ ]:
def dense_ae_score(df_window):
    '''Score Dense-AE sobre una ventana corta (DataFrame).'''
    Xf = compute_features(df_window, mode='full')
    X = scaler_dae.transform(Xf.values).astype('float32')
    if len(X) == 0:
        return np.array([], dtype='float32')
    rec = dae.predict(X, batch_size=512, verbose=0)
    return ((rec - X)**2 * weights_dae).sum(axis=1)


## 3. Ataques parametrizados por severidad continua

In [4]:
def attack_window(df_window, attack_type, severity, rng):
    '''Aplica un ataque con magnitud continua severity in [0, 1].'''
    df_atk = df_window.copy()
    gap_orig = df_atk['Global_active_power'].values.copy()
    g = gap_orig.copy()
    if attack_type == 'scaling':
        g = g * (1 - severity)
    elif attack_type == 'offset':
        g = g - severity * gap_orig.mean()
    elif attack_type == 'noise':
        g = g + rng.normal(0, severity * gap_orig.std() + 1e-6, len(g))
    elif attack_type == 'ramp':
        g = g - np.linspace(0, severity * gap_orig.mean(), len(g))
    elif attack_type == 'step':
        g = g - severity * gap_orig.mean()
    elif attack_type == 'replay':
        # Sustituir por valor medio + ruido (proxy de replay sin contexto historico)
        g = gap_orig.mean() * np.ones(len(g)) + rng.normal(0, gap_orig.std()*0.1*(1-severity), len(g))
    elif attack_type == 'voltage_spoof':
        df_atk['Voltage'] = df_atk['Voltage'] - severity * df_atk['Voltage'].std()
    df_atk['Global_active_power'] = np.maximum(g, 0)
    return df_atk


## 4. Búsqueda binaria del mínimo de evasión

In [5]:
def is_detected(df_window, attack_type, severity, threshold, rng):
    '''Devuelve True si el detector marca al menos 50% de los timesteps.'''
    atk = attack_window(df_window, attack_type, severity, rng)
    scores = dense_ae_score(atk)
    if len(scores) == 0:
        return False
    return (scores > threshold).mean() > 0.5    # mayoritariamente detectado


def find_min_evasion(df_window, attack_type, threshold, rng,
                      lo=0.01, hi=0.95, n_iter=12):
    '''Busqueda binaria: minima severidad detectada por el detector.'''
    # Sanity: severidad maxima debe ser detectada; minima no
    if not is_detected(df_window, attack_type, hi, threshold, rng):
        return None      # ni siquiera el ataque maximo es detectado -> indetectable
    if is_detected(df_window, attack_type, lo, threshold, rng):
        return lo        # la severidad minima ya es detectada -> muy robusto

    for _ in range(n_iter):
        mid = (lo + hi) / 2
        if is_detected(df_window, attack_type, mid, threshold, rng):
            hi = mid
        else:
            lo = mid
    return hi


## 5. Sampling de ventanas representativas

In [6]:
# Tomamos 30 ventanas aleatorias del set de TEST (zona limpia, sin ataques originales)
def sample_clean_windows(df, n_samples=30, win_len=120, seed=42):
    rng = np.random.default_rng(seed)
    # Zonas sin ataque
    clean_mask = (df['label']==0)
    valid_starts = np.where(clean_mask)[0]
    valid_starts = valid_starts[valid_starts < len(df) - win_len]
    chosen = rng.choice(valid_starts, size=n_samples, replace=False)
    return [df.iloc[s:s+win_len] for s in chosen]

t0 = time.time()
clean_windows = sample_clean_windows(df_test, n_samples=30, win_len=120, seed=42)
print(f'Muestreadas {len(clean_windows)} ventanas limpias en {time.time()-t0:.1f}s')


Muestreadas 30 ventanas limpias en 0.0s


## 6. Curva detection-rate vs severidad

In [7]:
ATTACK_TYPES_ADV = ['scaling', 'offset', 'noise', 'ramp', 'step', 'replay', 'voltage_spoof']
SEVERITIES_SWEEP = np.linspace(0.05, 0.90, 18)

print('Calculando curvas detection-rate vs severidad...')
results_curve = {}
rng = np.random.default_rng(2025)
for atype in ATTACK_TYPES_ADV:
    det_rate = []
    for sev in SEVERITIES_SWEEP:
        detected = 0
        for wnd in clean_windows:
            if is_detected(wnd, atype, sev, threshold_dae, rng):
                detected += 1
        det_rate.append(detected / len(clean_windows))
    results_curve[atype] = np.array(det_rate)
    print(f'  {atype:15s} - completado')

# Visualizacion
fig, ax = plt.subplots(figsize=(12, 6))
for atype, dr in results_curve.items():
    ax.plot(SEVERITIES_SWEEP, dr, marker='o', lw=2, label=atype)
ax.axhline(0.5, color='k', ls=':', alpha=0.5, label='50% threshold')
ax.set(xlabel='Severidad del ataque (epsilon)',
       ylabel='Detection rate',
       title='Robustez del Dense-AE — detection rate vs severidad')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, '..', 'figures', 'adversarial_curves.png'),
            dpi=120, bbox_inches='tight')
plt.show()


Calculando curvas detection-rate vs severidad...


ValueError: X has 14 features, but RobustScaler is expecting 17 features as input.

## 7. Mínimo de evasión por tipo de ataque (búsqueda binaria)

In [ ]:
print('Calculando epsilon* por tipo de ataque (binary search, 30 ventanas)...')
min_evasion = {}
for atype in ATTACK_TYPES_ADV:
    eps_list = []
    for wnd in clean_windows:
        eps = find_min_evasion(wnd, atype, threshold_dae, rng)
        if eps is not None:
            eps_list.append(eps)
    if len(eps_list) > 0:
        min_evasion[atype] = {
            'mean':   float(np.mean(eps_list)),
            'std':    float(np.std(eps_list)),
            'median': float(np.median(eps_list)),
            'n':      len(eps_list),
        }
    else:
        min_evasion[atype] = None
    if min_evasion[atype]:
        m = min_evasion[atype]
        print(f'  {atype:18s} eps* = {m["mean"]:.3f} +/- {m["std"]:.3f}  (n={m["n"]})')
    else:
        print(f'  {atype:18s} INDETECTABLE en todo el rango [0.01, 0.95]')

# Tabla resumen
rows = []
for atype, m in min_evasion.items():
    if m is None:
        rows.append({'tipo': atype, 'eps_mean': np.nan, 'eps_std': np.nan, 'evaluables': 0})
    else:
        rows.append({'tipo': atype, **{k: v for k,v in m.items() if k != 'mean'},
                     'eps_mean': m['mean']})
adv_df = pd.DataFrame(rows).set_index('tipo')
print('\n--- Tabla de robustez ---')
print(adv_df.round(3))


## 8. Interpretación económica

In [ ]:
# La severidad de scaling representa la fraccion de energia robada.
# Si el min eps* es 0.20, el atacante necesita robar al menos un 20% del consumo
# para superar el umbral del detector. Por debajo de eso pasa desapercibido.

print('='*65)
print('LECTURA ECONOMICA — Dense-AE')
print('='*65)
print()
if 'scaling' in min_evasion and min_evasion['scaling']:
    eps_sc = min_evasion['scaling']['mean']
    print(f'  Para scaling (robo proporcional):')
    print(f'  - Severity minima detectada: {eps_sc:.2%}')
    print(f'  - Significa: un atacante que robe MENOS del {eps_sc:.0%} de la')
    print(f'    energia pasa desapercibido para este detector.')
    print(f'  - En un consumo medio de 670 W, eso equivale a robar')
    print(f'    {670*eps_sc:.1f} W = {670*eps_sc*24/1000:.2f} kWh/dia.')
    print()
    if eps_sc < 0.10:
        print('  Interpretacion: detector FRAGIL — ventana evasion grande.')
    elif eps_sc < 0.25:
        print('  Interpretacion: detector MODERADO — atacante necesita ser cauteloso.')
    else:
        print('  Interpretacion: detector ROBUSTO — robo de bajo ratio es economicamente inviable.')

if 'voltage_spoof' in min_evasion and min_evasion['voltage_spoof']:
    print()
    eps_vs = min_evasion['voltage_spoof']['mean']
    print(f'  Para voltage_spoof (spoofing fisico):')
    print(f'  - Eps minimo detectado: {eps_vs:.2%}')
    print(f'  - VI_residual es la feature clave: cualquier spoofing de >{eps_vs:.0%}σ')
    print(f'    del voltaje rompe la invariante fisica y dispara la deteccion.')


## 9. Guardado

In [ ]:
results = {
    'min_evasion_per_attack': {k: v for k,v in min_evasion.items()},
    'detection_rate_curves': {k: v.tolist() for k,v in results_curve.items()},
    'severities_sweep': SEVERITIES_SWEEP.tolist(),
    'window_length_min': 120,
    'n_windows_sampled': 30,
    'detector': 'Dense-AE',
    'threshold_used': float(threshold_dae),
}
with open(os.path.join(DATA_DIR, 'adversarial_robustness.json'), 'w') as f:
    json.dump(results, f, indent=2)
adv_df.to_csv(os.path.join(DATA_DIR, 'adversarial_min_evasion.csv'))
print('Guardado:')
print('  data/adversarial_robustness.json')
print('  data/adversarial_min_evasion.csv')
print('  figures/adversarial_curves.png')


## 10. Conclusión

In [ ]:
print('='*65)
print('NB14 — ROBUSTEZ ADVERSARIA: CONCLUSIONES')
print('='*65)
print()
print('1. Para cada tipo de ataque calculamos epsilon*, el umbral minimo')
print('   de severidad que el detector consigue marcar como anomalia.')
print()
print('2. La curva detection-rate vs severidad cuantifica el riesgo:')
print('   un atacante racional eligira severidad apenas inferior a epsilon*')
print('   para maximizar robo manteniendo no-deteccion.')
print()
print('3. Trabajo futuro: ataques GRADIENT-based (FGSM, PGD adaptados a')
print('   redes con score = MSE) podrian encontrar evasiones aun menores')
print('   conociendo los pesos del detector.')
print()
print('4. Mitigacion: ENSEMBLE DIVERSO (NB10 Stacking) **reduce el espacio')
print('   de evasion porque el atacante tendria que evadir a 6 modelos a la vez**.')
print('   Este es el argumento academico final de defensa del Stacking.')
